# Episode 12: Give Your Agents a Knowledge Base

Your agents are smart, but they only know what the LLM was trained on. What about your company's docs, your policies, your data?

In this episode, you'll build an agent that answers questions by searching your documents, not just using LLM knowledge.

## What is RAG?

**Retrieval-Augmented Generation (RAG)** solves a fundamental problem: LLMs only know what was in their training data. They can't answer questions about your company's internal docs, your product specs, or last week's meeting notes. RAG is consistently the most requested enterprise AI feature — and for good reason.

RAG fixes this by adding a search step before the LLM generates its answer:

```
Question → Search docs → Find relevant chunks → Feed to LLM → Answer
```

The LLM gets the relevant context *at query time*, so it can answer questions about information it was never trained on.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen import AssistantAgent, LLMConfig
from autogen.agentchat import run_group_chat
from autogen.agentchat.group.patterns import DefaultPattern
from autogen.agentchat.group import TerminateTarget

llm_config = LLMConfig({"model": "gpt-4o-mini"})

## Step 1: Create sample documents

Now that you understand the pattern, let's build one. We'll create a small knowledge base for a fictional company called **NovaTech**. In a real system, these would be loaded from files, a database, or a document store. Let's see this:

In [ ]:
# Our knowledge base: a list of documents, each with a title and content
knowledge_base = [
    {
        "title": "NovaTech Products",
        "content": (
            "NovaTech offers three main products:\n"
            "1. NovaDB - A distributed database optimized for real-time analytics. "
            "Supports SQL and GraphQL queries. Pricing starts at $99/month for the Starter plan, "
            "$299/month for Pro, and custom pricing for Enterprise.\n"
            "2. NovaFlow - A workflow automation platform for building data pipelines. "
            "Includes 50+ pre-built connectors. Free tier available with up to 1000 runs/month.\n"
            "3. NovaGuard - A security monitoring tool that uses AI to detect anomalies. "
            "Integrates with AWS, GCP, and Azure. Starting at $199/month."
        ),
    },
    {
        "title": "NovaTech Policies",
        "content": (
            "Return Policy: All subscriptions come with a 30-day money-back guarantee. "
            "Annual plans can be canceled within 14 days for a full refund.\n"
            "Support Policy: Starter plans include email support (48hr response). "
            "Pro plans include priority email and chat support (4hr response). "
            "Enterprise plans include 24/7 phone support with a dedicated account manager.\n"
            "SLA: NovaTech guarantees 99.9% uptime for Pro and Enterprise plans. "
            "Downtime credits are issued automatically if the SLA is not met."
        ),
    },
    {
        "title": "NovaTech Team",
        "content": (
            "Leadership Team:\n"
            "- CEO: Maria Chen, former VP of Engineering at DataCorp. Founded NovaTech in 2023.\n"
            "- CTO: James Okafor, PhD in distributed systems from MIT. "
            "Led the development of NovaDB.\n"
            "- VP of Sales: Sarah Kim, 15 years of enterprise SaaS experience.\n"
            "- Head of AI: David Park, previously at DeepMind. "
            "Leading the NovaGuard AI anomaly detection team.\n"
            "The company has 150 employees across San Francisco, London, and Singapore."
        ),
    },
]

print(f"Knowledge base loaded: {len(knowledge_base)} documents")
for doc in knowledge_base:
    print(f"  - {doc['title']}")

## Step 2: Build a simple retrieval tool

Now that you've seen the documents, we need a way to search them. This function searches the knowledge base by keyword matching — it splits each document into chunks and returns chunks that contain any of the query's keywords.

This is simplified RAG — no vector database or embeddings needed for this demo. Let's see this:

In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for information relevant to the query.

    Args:
        query: The search query -- use specific keywords for best results.

    Returns:
        Matching document chunks, or a message if nothing was found.
    """
    query_words = set(query.lower().split())
    results = []

    for doc in knowledge_base:
        # Search in both title and content
        chunks = doc["content"].split("\n")
        for chunk in chunks:
            chunk_lower = chunk.lower()
            # Match if any query word appears in the chunk
            if any(word in chunk_lower for word in query_words if len(word) > 2):
                results.append(f"[{doc['title']}] {chunk.strip()}")

    if results:
        return "\n\n".join(results[:5])  # Return top 5 matches
    return "No relevant information found in the knowledge base."


# Quick test
print("Search for 'CEO':")
print(search_knowledge_base("CEO"))
print()
print("Search for 'return policy':")
print(search_knowledge_base("return policy"))

## Step 3: Create a RAG agent

This leads to the crucial design decision: the agent's system message must instruct it to **always search the knowledge base before answering**. This prevents the agent from hallucinating information about our fictional company. Let's see this:

In [ ]:
knowledge_agent = AssistantAgent(
    name="knowledge_agent",
    system_message=(
        "You are a NovaTech support agent with access to the company knowledge base.\n\n"
        "IMPORTANT RULES:\n"
        "1. ALWAYS search the knowledge base before answering a question.\n"
        "2. Only provide information that is found in the knowledge base.\n"
        "3. If the knowledge base doesn't have the answer, say so honestly.\n"
        "4. Cite which document the information came from.\n"
        "5. Be concise and helpful."
    ),
    llm_config=llm_config,
)

user = AssistantAgent(name="user", human_input_mode="NEVER")

# LLM agent suggests the search, user agent executes it
knowledge_agent.register_for_llm()(search_knowledge_base)
user.register_for_execution()(search_knowledge_base)

## Step 4: Test it

Now that you've seen the agent and its tool, let's ask questions that require information from our knowledge base. Notice how the agent searches first, then answers based on what it finds — and cites the source. It didn't hallucinate the answer.

In [ ]:
knowledge_agent.set_after_work(TerminateTarget())

pattern = DefaultPattern(
    initial_agent=knowledge_agent,
    agents=[knowledge_agent],
)

# Question 1: About people
print("=" * 60)
print("Question 1: Who is the CEO?")
print("=" * 60)
result = run_group_chat(
    pattern=pattern,
    messages="Who is the CEO of NovaTech?",
    max_rounds=6,
)
result.process()

In [ ]:
# Question 2: About policies
print("=" * 60)
print("Question 2: What is the return policy?")
print("=" * 60)
result = run_group_chat(
    pattern=pattern,
    messages="What is the return policy for NovaTech subscriptions?",
    max_rounds=6,
)
result.process()

In [ ]:
# Question 3: About products
print("=" * 60)
print("Question 3: How much does NovaDB cost?")
print("=" * 60)
result = run_group_chat(
    pattern=pattern,
    messages="How much does NovaDB cost? What plans are available?",
    max_rounds=6,
)
result.process()

## Vector search vs keyword search: trade-offs

Our simple keyword search works for demos, but there are clear trade-offs. It can't understand synonyms or semantic meaning — "Who founded the company?" might miss the answer because the matching is purely lexical. It also breaks down with larger document collections.

In production, you'll want to choose the approach that fits your data and query patterns:

| Approach | How it works | Best for |
|----------|-------------|----------|
| **Keyword search** | Exact/partial string matching | Small docs, prototyping |
| **Vector search** | Embed text as vectors, find nearest neighbors | Large docs, semantic queries |
| **Graph RAG** | Store facts as knowledge graph nodes + edges | Structured relationships |

Popular vector databases: **ChromaDB** (simple, local), **Qdrant** (production-ready), **pgvector** (if you already use PostgreSQL). For Graph RAG, **FalkorDB** provides a graph database that agents can query for structured knowledge — useful when your data has complex relationships (org charts, product dependencies, etc.).

## What RAG agents need

A production RAG system typically gives the agent several specialized tools. Here's what a complete setup looks like:

| Tool | Purpose |
|------|----------|
| **Search/retrieve** | Find relevant documents by query |
| **Chunk** | Split large documents into manageable pieces |
| **Embed** | Convert text to vector representations |
| **Rerank** | Re-order results by relevance after initial retrieval |

AG2 has built-in RAG support — see the [RetrieveChat blog post](https://docs.ag2.ai/latest/docs/blog/2023/10/18/RetrieveChat) for how to use `RetrieveUserProxyAgent` with various vector databases. For a Graph RAG example using FalkorDB, see the `travel-planner/` project in this workshop.

## Try it yourself

What happens if a question spans multiple documents? For example, "What support level does a Pro plan NovaDB customer get?" requires both the Products and Policies docs.

Try these experiments:

1. **Add more documents** to the knowledge base — try adding a "NovaTech Roadmap" or "FAQ" document.
2. **Ask a cross-document question** like the one above and see how the agent handles it.
3. **Improve the search** — try making `search_knowledge_base` smarter by adding stemming, removing stop words, or scoring by match count.

In [ ]:
# Your code here

## What's next?

Your agents can now search your documents — but what about information that changes every day? What if they could browse the live web?